In [15]:
import sys
import pandas as pd
import os
from pathlib import Path

# The below code is used to access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))


In [16]:
print(os.getcwd())

c:\Users\USER\Documents\AI projects\DenseLess\app\agent


In [17]:
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma
from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings
from langchain_core.documents import Document
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from typing import List
from app.agent.rag.chains.qa_chain import run_qa_chain, list_conversations, view_ltm
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import json

In [18]:
def format_chunks(chunks: List[Document]) -> str:
    if not chunks:
        return ""
    return "\n".join(f"{i + 1}. {doc.page_content}" for i, doc in enumerate(chunks))

In [20]:
load_dotenv()
api_key = os.environ.get("GOOGLE_API_KEY")

embedder = get_embedding_model()
store = get_vector_store(student_id="1019", embedder=embedder)

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BA

  → Embedding model loaded successfully. Embedding dimension: 384
Vector store backend: 'chroma' | student: '1019'
Initialising Chroma store — student: '1019'
  → Collection : student_1019
  → Persist dir: ../chroma_db
  → Chroma store ready. 
Documents in collection: 337


In [21]:

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    project="denseless",
    location="us-central1",
    vertexai=True,
)

questions = [
    "What is the difference between volatile and nonvolatile memory, and how does read-only memory (ROM) fit into these categories?"
]   


INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [22]:
json_eval_dataset = {}

for i, q in enumerate(questions, start=1):
    key = str(i).zfill(3)

    chunks = get_semantic_chunks(
        query=q,
        store=store,
        score_threshold=0.4,
        top_k=5,
        topic="Learning Memory System",
        course="CSC 315",
        condensed=False,
    )
    formatted_chunks = format_chunks(chunks)

    response = run_qa_chain(
        "student_1019",
        q,
        "COA_chat_3",
        False,
        store,
        llm,
        "fast",
        "Learning Memory System.pdf",
        "CSC 315",
    )
    qa_output = response.content["answer"]

    json_eval_dataset[key] = {
        "student_question": q,
        "source_chunks": formatted_chunks,
        "qa_output": qa_output,
    }


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'What is the difference between volatile and nonvolatile memo'
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 5 chunk(s), 1 passed threshold.
[Retriever] → 1 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 1 section(s): {'3.1 Memory Characteristics'}
[Retriever]   → Section '3.1 Memory Characteristics': 24 chunk(s) fetched.
[Retriever] → Expansion complete: 1 seed → 24 unique chunk(s) after deduplication.
[Retriever] → Final result: 24 chunk(s) returned to chain.
  [token_service] Tokens remaining for 'student_1019': 9,523,946
  [token_guard] Checking tokens — student: student_1019 | remaining: 9523946 | chain: run_qa_chain
[qa_chain] Starting QA — student: 'student_1019' | topic: 'Learning Memory System.pdf' | convo: 'COA_chat_3' | pace: 'fast'
[qa_chain] STM loaded for 'COA_ch

INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[qa_chain] Question reformulated.
[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'What is the difference between volatile and nonvolatile memo'
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 5 chunk(s), 1 passed threshold.
[Retriever] → 1 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 1 section(s): {'3.1 Memory Characteristics'}
[Retriever]   → Section '3.1 Memory Characteristics': 24 chunk(s) fetched.
[Retriever] → Expansion complete: 1 seed → 24 unique chunk(s) after deduplication.
[Retriever] → Final result: 24 chunk(s) returned to chain.
[qa_chain] Retrieved 24 chunk(s).
1. 3.1 Memory Characteristics
2. 1. Location: Refers to whether memory is internal or external to the computer Internal memory is often equated with main memory, but there are other forms of internal memory; Processor requires its own local memory in th

INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[qa_chain] QA LLM call succeeded. Grounded: None


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[qa_chain] STM summary updated (4906 chars).
[qa_chain] STM saved for 'COA_chat_3' (17 turns summarised).


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[qa_chain] LTM raw manager output: [ExtractedMemory(id='9381b7b0-db0e-4c73-928e-303f306ae37f', content=Memory(content='The student has demonstrated a lack of understanding regarding the classification of Read-Only Memory (ROM) within the volatile and nonvolatile memory categories.'))]
[qa_chain] LTM store saved for student 'student_1019' (1 memories).
[qa_chain] LTM: 1 new memory/memories extracted and saved.
[qa_chain] QA chain complete for student 'student_1019'.
  [token_service] Deducted 3,609 tokens — student: 'student_1019' | chain: run_qa_chain | in: 3,507 | out: 102 | remaining: 9,520,337
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 3609 | remaining: 9520337


In [23]:
output_path = Path(os.getcwd()) / "rag" / "eval_data" / "eval_dataset.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(json_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"Successfully completed. Exported to {output_path}")


Successfully completed. Exported to c:\Users\USER\Documents\AI projects\DenseLess\app\agent\rag\eval_data\eval_dataset.json
